# 08 · Proxy exploratorio de carbono forestal

**Objetivo:** Combinar cobertura, NDVI y pendiente para priorizar inventarios.

**Datos:** Sentinel-2, Dynamic World y SRTM.

**Relevancia para política ambiental y social:** Ayuda a focalizar trabajo de campo en proyectos forestales.

**Limitaciones:** No cuantifica existencias ni créditos de carbono y no sustituye inventarios.


In [ ]:
# Instalar dependencias en Google Colab
!pip -q install earthengine-api geemap

import ee
import geemap
import datetime

ee.Authenticate()
ee.Initialize(project="TU_PROYECTO_GEE")

# Área de estudio de ejemplo: entorno de Chachapoyas, Amazonas, Perú
# Reemplázala por un polígono, activo de Earth Engine o coordenadas propias.
aoi = ee.Geometry.Point([-77.87, -6.23]).buffer(30000)

Map = geemap.Map()
Map.centerObject(aoi, 9)


In [ ]:
def mask_s2_sr(image):
    scl = image.select("SCL")
    clear = (
        scl.neq(3)   # sombra
        .And(scl.neq(8))  # nube media
        .And(scl.neq(9))  # nube alta
        .And(scl.neq(10)) # cirrus
        .And(scl.neq(11)) # nieve/hielo
    )
    return image.updateMask(clear).divide(10000).copyProperties(
        image, ["system:time_start"]
    )

def s2_composite(start, end):
    return (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40))
        .map(mask_s2_sr)
        .median()
        .clip(aoi)
    )


In [ ]:
s2 = s2_composite("2025-01-01","2025-12-31")
ndvi = s2.normalizedDifference(["B8","B4"]).unitScale(0.2,0.9).clamp(0,1)
forest = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterBounds(aoi).filterDate("2025-01-01","2025-12-31").select("label").mode().eq(1)
slope = ee.Terrain.slope(ee.Image("USGS/SRTMGL1_003")).unitScale(0,45).clamp(0,1)
proxy = ndvi.multiply(0.6).add(forest.multiply(0.3)).add(slope.multiply(0.1)).rename("priority").clip(aoi)

Map.addLayer(proxy, {"min":0,"max":1,"palette":["yellow","green","darkgreen"]}, "Prioridad de inventario")
Map
